<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_06_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_06 - T2 SEQ2ONE - Transformer**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-22 21:21:03,253 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

Mounted at /content/drive


2026-04-22 21:21:25,859 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-22 21:21:27,936 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-22 21:21:27,937 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-22 21:21:27,938 | INFO | Configuración de experimento cargada
2026-04-22 21:21:27,938 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-22 21:21:27,939 | INFO | Window sizes: [30]


In [4]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-22 21:21:27,949 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-22 21:21:29,567 | INFO | Windows OK      : 9
2026-04-22 21:21:29,568 | INFO | Windows missing : 0
2026-04-22 21:21:29,569 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-22 21:21:29,569 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [5]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [6]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [7]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [8]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [9]:
bundles_L30 = create_bundles(window_size=30)

2026-04-22 21:21:30,742 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-22 21:21:30,743 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-22 21:21:31,671 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-22 21:21:31,672 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-22 21:21:32,711 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-22 21:21:32,712 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-22 21:21:34,612 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-22 21:21:34,613 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-22 21:21:36,594 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-22 21:21:36,594 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-22 21:21:37,498 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-22 21:21:37,498 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-22 21:21:38,390 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




In [10]:
bundles_L30

{'t2_p40_h30': {'window_size': 30,
  'target': 't2_p40_h30',
  'horizon': 30,
  'paths': {'train': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_train.npz',
   'valid': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_valid.npz',
   'test': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_test.npz',
   'scaler': '/content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl'},
  'scaler': StandardScaler(),
  'train': {'X': array([[[-0.11680385, -0.06583842, -0.2238865 , ..., -0.04129868,
            -1.507039  , -0.02778307],
           [ 0.0909589 ,  0.05818945, -0.07248875, ...,  0.3872164 ,
            -1.3723946 ,  0.08631181],
           [-0.18229802, -0.13053213, -0.21077001, ..., -0.1151728 ,
            -1.2194692 ,  0.02307655],
           ...,
           [ 0.40151003,  0.28442496,  0.59289074, ...,  0.47497907,
            -1.0690751 , -0.19294474],
         

### **4.5. Preparación de inputs según el tipo de modelo**

In [11]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [12]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [13]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | c

## **6. Módulo de métricas T2**

In [14]:
# ================================
# Setup para importar módulos del proyecto
# ================================

import sys
import importlib

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

# asegurar que metrics es paquete
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from metrics.classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")

Módulos importados correctamente


In [15]:
# ================================
# Utilidades: outputs -> DataFrame
# ================================

from __future__ import annotations

import pandas as pd
from typing import Any


def classification_metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output de compute_classification_metrics(...)
    en una fila de DataFrame.

    Usa metrics_to_flat_dict(...) para aplanar la salida
    del módulo classification_metrics y luego agrega metadata
    del experimento.
    """
    flat_metrics = metrics_to_flat_dict(metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_metrics,
    }

    return pd.DataFrame([row])


def _flatten_dict(
    d: dict[str, Any],
    *,
    parent_key: str = "",
    sep: str = "_",
) -> dict[str, Any]:
    """
    Aplana un diccionario arbitrario de forma recursiva.

    Ejemplo:
    {"a": {"b": 1}} -> {"a_b": 1}
    """
    items: dict[str, Any] = {}

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(_flatten_dict(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v

    return items


def probabilities_metrics_to_df(
    prob_metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output del módulo classification_probabilities
    en una fila de DataFrame.

    Como la estructura puede variar según la implementación,
    se aplana recursivamente el diccionario y luego se agrega
    metadata del experimento.
    """
    flat_prob_metrics = _flatten_dict(prob_metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_prob_metrics,
    }

    return pd.DataFrame([row])


logger.info("Utilidades de exportación a DataFrame cargadas")

2026-04-22 21:21:45,944 | INFO | Utilidades de exportación a DataFrame cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [16]:
# ================================
# Persistencia de métricas de clasificación
# ================================

from pathlib import Path
import pandas as pd


def load_classification_metrics_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen métricas previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path


# ================================
# Persistencia de probabilidades / decisión
# ================================

def load_classification_probabilities_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_probabilities",
) -> pd.DataFrame:
    """
    Carga resultados probabilísticos si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando probabilidades desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen probabilidades previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_probabilities(
    df_probabilities: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics" / "classification_probabilities",
) -> Path:
    """
    Guarda un DataFrame de probabilidades / decisión en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"
    df_probabilities.to_parquet(out_path, index=False)

    logger.info(f"Probabilidades guardadas en: {out_path}")

    return out_path

Ejemplo de uso con logistic regression:

```python
df_metrics_all = load_classification_metrics_if_exists(
    model_name="logistic_regression",
    split="valid",
)

df_probabilities_all = load_classification_probabilities_if_exists(
    model_name="logistic_regression",
    split="valid",
)
```

Guardar:
```python
save_classification_metrics(
    df_metrics_all,
    model_name="logistic_regression",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_all,
    model_name="logistic_regression",
    split="valid",
)
```

## **8. Gestión de dispositivo y memoria**

In [17]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [18]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-22 21:21:50,353 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo**

## **10.1. Función unitaria por bundle**

In [30]:
import copy
import math
import numpy as np
import torch
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)

        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)


class TransformerClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        d_model: int = 32,
        nhead: int = 4,
        num_layers: int = 1,
        dim_feedforward: int = 64,
        dropout: float = 0.0,
        num_classes: int = 3,
        max_len: int = 512,
    ):
        super().__init__()

        if d_model % nhead != 0:
            raise ValueError(
                f"d_model ({d_model}) debe ser divisible por nhead ({nhead})"
            )

        # Proyección inicial desde n_features hacia d_model
        self.input_proj = nn.Linear(n_features, d_model)

        # Positional encoding para incorporar información temporal
        self.pos_encoder = PositionalEncoding(
            d_model=d_model,
            max_len=max_len,
            dropout=dropout,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        """
        x: (batch, seq_len, n_features)
        """
        x = self.input_proj(x)        # (batch, seq_len, d_model)
        x = self.pos_encoder(x)
        out = self.encoder(x)         # (batch, seq_len, d_model)
        last_out = out[:, -1, :]      # many-to-one
        last_out = self.dropout(last_out)
        logits = self.fc(last_out)
        return logits


def run_transformer_for_bundle_seq2one(
    bundle,
    *,
    d_model: int = 32,
    nhead: int = 4,
    num_layers: int = 1,
    dim_feedforward: int = 64,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    deterministic: bool = True,
    class_weight=None,
    num_workers: int = 0,
    optimizer_name: str = "adam",
    grad_clip_norm: float | None = None,
    verbose: bool = False,
):
    """
    Ejecuta Transformer para un bundle seq2one.
    Evalúa SOLO sobre VALID.

    Enfoque actual del pipeline:
    - TRAIN para entrenamiento
    - VALID para early stopping y evaluación
    - sin uso de TEST en esta etapa
    - soporte para labels arbitrarias mediante codificación interna
    - uso de GPU si está disponible
    """

    # =========================
    # 1. SEEDS Y DEVICE
    # =========================
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    use_pin_memory = device == "cuda"

    # =========================
    # 2. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 3. VALIDAR SHAPES
    # =========================
    if X_train.ndim != 3 or X_valid.ndim != 3:
        raise ValueError(
            "Transformer requiere tensores 3D: (n_samples, seq_len, n_features). "
            f"Recibido train={X_train.shape}, valid={X_valid.shape}"
        )

    seq_len_train, n_features_train = X_train.shape[1], X_train.shape[2]
    seq_len_valid, n_features_valid = X_valid.shape[1], X_valid.shape[2]

    if not (
        seq_len_train == seq_len_valid
        and n_features_train == n_features_valid
    ):
        raise ValueError(
            "Inconsistencia entre shapes de train/valid. "
            f"train={X_train.shape}, valid={X_valid.shape}"
        )

    n_features = n_features_train
    seq_len = seq_len_train

    # =========================
    # 4. CODIFICAR LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))

    unknown_valid = set(np.unique(y_valid)) - set(classes_)
    if unknown_valid:
        raise ValueError(
            f"VALID contiene clases no vistas en TRAIN: {sorted(unknown_valid)}"
        )

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int64)
    y_valid_enc = np.array([class_to_idx[y] for y in y_valid], dtype=np.int64)

    num_classes = len(classes_)

    # =========================
    # 5. CLASS WEIGHTS
    # =========================
    criterion_weight = None
    weights_by_idx = None

    if class_weight is None:
        criterion_weight = None

    elif class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_classes)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_classes * count)
            for idx, count in enumerate(counts)
        }

        criterion_weight = torch.tensor(
            [weights_by_idx[idx] for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        criterion_weight = torch.tensor(
            [weights_by_idx.get(idx, 1.0) for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    else:
        raise ValueError("class_weight debe ser None, 'balanced' o dict")

    # =========================
    # 6. TENSORES EN CPU
    # =========================
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_enc, dtype=torch.long)

    X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
    y_valid_t = torch.tensor(y_valid_enc, dtype=torch.long)

    # =========================
    # 7. DATALOADERS
    # =========================
    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    valid_ds = torch.utils.data.TensorDataset(X_valid_t, y_valid_t)

    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    valid_loader = torch.utils.data.DataLoader(
        valid_ds,
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    # =========================
    # 8. MODELO
    # =========================
    model = TransformerClassifier(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
        num_classes=num_classes,
        max_len=max(seq_len, 512),
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=criterion_weight)

    if optimizer_name.lower() == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
    elif optimizer_name.lower() == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
        )
    else:
        raise ValueError(f"optimizer_name no soportado: {optimizer_name}")

    # =========================
    # 9. HELPERS
    # =========================
    def _move_batch(x):
        if device == "cuda":
            return x.to(device, non_blocking=True)
        return x.to(device)

    def compute_valid_loss():
        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0

        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = _move_batch(xb)
                yb = _move_batch(yb)

                logits = model(xb)
                loss = criterion(logits, yb)

                batch_n = xb.size(0)
                valid_loss_sum += loss.item() * batch_n
                valid_count += batch_n

        return valid_loss_sum / max(valid_count, 1)

    def predict_loader(loader):
        logits_all = []

        model.eval()
        with torch.no_grad():
            for xb, *_ in loader:
                xb = _move_batch(xb)
                logits = model(xb)
                logits_all.append(logits.cpu())

        logits_all = torch.cat(logits_all, dim=0)
        return logits_all

    # =========================
    # 10. TRAIN + EARLY STOPPING
    # =========================
    best_state = None
    best_valid_loss = np.inf
    best_epoch = 0
    wait = 0
    history = []

    try:
        for epoch in range(1, epochs + 1):
            model.train()
            train_loss_sum = 0.0
            train_count = 0

            for xb, yb in train_loader:
                xb = _move_batch(xb)
                yb = _move_batch(yb)

                optimizer.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = criterion(logits, yb)
                loss.backward()

                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

                optimizer.step()

                batch_n = xb.size(0)
                train_loss_sum += loss.item() * batch_n
                train_count += batch_n

            train_loss = train_loss_sum / max(train_count, 1)
            valid_loss = compute_valid_loss()

            history.append(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "valid_loss": valid_loss,
                }
            )

            if verbose:
                print(
                    f"[Epoch {epoch:03d}] "
                    f"train_loss={train_loss:.6f} | "
                    f"valid_loss={valid_loss:.6f}"
                )

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(
                            f"[EARLY STOP] epoch={epoch} | "
                            f"best_epoch={best_epoch} | "
                            f"best_valid_loss={best_valid_loss:.6f}"
                        )
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        # =========================
        # 11. PREDICT (VALID)
        # =========================
        valid_logits = predict_loader(valid_loader)

        y_pred_valid_enc = valid_logits.argmax(dim=1).numpy()
        y_proba_valid = torch.softmax(valid_logits, dim=1).numpy()
        y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])

        if verbose:
            print(
                f"[TRANSFORMER] target={target} | horizon={horizon} | "
                f"window_size={window_size} | device={device} | "
                f"X_train={X_train.shape} | X_valid={X_valid.shape} | "
                f"best_epoch={best_epoch} | best_valid_loss={best_valid_loss:.6f}"
            )

        return {
            "model_name": "transformer",
            "target": target,
            "horizon": horizon,
            "window_size": window_size,
            "class_weight": class_weight,
            "optimizer_name": optimizer_name,
            "grad_clip_norm": grad_clip_norm,
            "model": model,
            "classes_": classes_,
            "class_to_idx": class_to_idx,
            "idx_to_class": idx_to_class,
            "criterion_weight": (
                criterion_weight.detach().cpu().numpy()
                if criterion_weight is not None else None
            ),
            "weights_by_idx": weights_by_idx,
            "history": history,
            "best_valid_loss": float(best_valid_loss),
            "best_epoch": int(best_epoch),
            "device": device,
            "y_valid": y_valid,
            "y_pred_valid": y_pred_valid,
            "y_proba_valid": y_proba_valid,
        }

    finally:
        if device == "cuda":
            torch.cuda.empty_cache()

## **10.2. Función de evaluación sobre uno o más bundles**

In [33]:
from typing import Any, Dict, List, Sequence, Union
import gc
import pandas as pd
import torch


from typing import Any, Dict, List, Sequence, Union
import gc
import pandas as pd
import torch


def eval_transformer_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "transformer",
    d_model: int = 32,
    nhead: int = 4,
    num_layers: int = 1,
    dim_feedforward: int = 64,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
    num_workers: int = 0,
    optimizer_name: str = "adam",
    grad_clip_norm: float | None = None,
    prob_threshold_long: float = 0.4,
    prob_threshold_short: float = 0.4,
    verbose: bool = False,
):
    # --------------------------------------------------
    # 1) Normalizar entrada
    # --------------------------------------------------
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    rows_metrics = []
    rows_probs = []


    # ----------------------------
    # 2) Loop bundles
    # ----------------------------
    for bundle in bundles_list:
        target = bundle.get("target")
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(f"-> Transformer | L{window_size} | target={target}")

        preds = None

        try:
            # ----------------------------
            # 3) Train + predict
            # ----------------------------
            preds = run_transformer_for_bundle_seq2one(
                bundle,
                d_model=d_model,
                nhead=nhead,
                num_layers=num_layers,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                learning_rate=learning_rate,
                weight_decay=weight_decay,
                batch_size=batch_size,
                eval_batch_size=eval_batch_size,
                epochs=epochs,
                patience=patience,
                random_state=random_state,
                device=device,
                class_weight=class_weight,
                num_workers=num_workers,
                optimizer_name=optimizer_name,
                grad_clip_norm=grad_clip_norm,
                verbose=False,
            )

            # ----------------------------
            # 4) Datos VALID
            # ----------------------------
            y_true = preds["y_valid"]
            y_pred = preds["y_pred_valid"]
            y_proba = preds["y_proba_valid"]
            classes_ = preds["classes_"]

            # ----------------------------
            # 5) Métricas clasificación
            # ----------------------------
            metrics = compute_classification_metrics(
                y_true=y_true,
                y_pred=y_pred,
                model_name=model_name,
                split="valid",
                target=target,
                labels=[-1, 0, 1],
            )

            df_metrics = classification_metrics_to_df(
                metrics,
                model=model_name,
                split="valid",
                window_size=window_size,
                target=target,
                horizon=horizon,
            )

            df_metrics["class_weight_mode"] = class_weight
            df_metrics["optimizer_name"] = optimizer_name
            df_metrics["grad_clip_norm"] = grad_clip_norm

            rows_metrics.append(df_metrics)

            # ----------------------------
            # 6) Probabilidades
            # ----------------------------
            proba_df = compute_probabilistic_outputs(
                y_proba=y_proba,
                class_labels=classes_,
                y_true=y_true,
            )

            proba_df = apply_decision_rule(
                proba_df,
                long_class=1,
                short_class=-1,
                long_threshold=prob_threshold_long,
                short_threshold=prob_threshold_short,
            )

            # metadata
            proba_df["model"] = model_name
            proba_df["target"] = target
            proba_df["horizon"] = horizon
            proba_df["class_weight_mode"] = class_weight
            proba_df["optimizer_name"] = optimizer_name
            proba_df["grad_clip_norm"] = grad_clip_norm
            proba_df["threshold_long"] = prob_threshold_long
            proba_df["threshold_short"] = prob_threshold_short

            rows_probs.append(proba_df)

        finally:
            if preds is not None:
                del preds
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # ----------------------------
    # 7) Output final
    # ----------------------------
    df_metrics_all = pd.concat(rows_metrics, ignore_index=True)
    df_probabilities_all = pd.concat(rows_probs, ignore_index=True)

    return {
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **10.3. Función orquestadora por `window_size`**

In [39]:
import gc
import pandas as pd
import torch


def run_transformer(
    window_size: int,
    *,
    targets,
    verbose: bool = True,
    model_name: str = "transformer",
    d_model: int = 32,
    nhead: int = 4,
    num_layers: int = 1,
    dim_feedforward: int = 64,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 32,
    eval_batch_size: int = 32,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
    num_workers: int = 0,
    optimizer_name: str = "adam",
    grad_clip_norm: float | None = None,
    prob_threshold_long: float = 0.4,
    prob_threshold_short: float = 0.4,
):
    """
    Orquestador Transformer (PIPELINE FINAL)

    - SOLO VALID
    - Devuelve:
        df_metrics_all
        df_probabilities_all
    """

    size = int(window_size)

    model_name_effective = (
        f"{model_name}_balanced" if class_weight == "balanced" else model_name
    )

    try:
        # --------------------------------------------------
        # 1) Header
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"TRANSFORMER | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"targets       = {targets}")
            print(f"class_weight  = {class_weight}")
            print(f"d_model       = {d_model}")
            print(f"nhead         = {nhead}")
            print(f"batch_size    = {batch_size}")
            print(f"optimizer     = {optimizer_name}")
            print(f"grad_clip     = {grad_clip_norm}")
            print(f"thr_long      = {prob_threshold_long}")
            print(f"thr_short     = {prob_threshold_short}")

        # --------------------------------------------------
        # 2) Bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | n_targets={len(targets)}")

        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # --------------------------------------------------
        # 3) Evaluación (VALID ONLY)
        # --------------------------------------------------
        if verbose:
            print(f"\n[EVAL] VALID | model={model_name_effective}")

        results = eval_transformer_bundles(
            bundles,
            model_name=model_name_effective,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            eval_batch_size=eval_batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            num_workers=num_workers,
            optimizer_name=optimizer_name,
            grad_clip_norm=grad_clip_norm,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        df_metrics_all = results["metrics"]
        df_probabilities_all = results["probabilities"]

        # --------------------------------------------------
        # 4) Orden
        # --------------------------------------------------
        df_metrics_all = (
            df_metrics_all
            .sort_values(["target", "horizon"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size}")
            print(df_metrics_all[
                [
                    "target",
                    "horizon",
                    "balanced_accuracy",
                    "f1_macro",
                ]
            ].to_string(index=False))

        return {
            "metrics": df_metrics_all,
            "probabilities": df_probabilities_all,
        }

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## **10.4. Ejecución final del experimento**

In [40]:
results_transformer = run_transformer(
    window_size=30,
    targets=["t2_p40_h30", "t2_p40_h60", "t2_p50_h30"],
    model_name="transformer",
    d_model=32,
    nhead=4,
    num_layers=1,
    dim_feedforward=64,
    batch_size=512,
    eval_batch_size=1024,
    epochs=20,
    patience=5,
    class_weight="balanced",
    optimizer_name="adamw",
    grad_clip_norm=1.0,
    prob_threshold_long=0.40,
    prob_threshold_short=0.40,
    device="cuda",
    verbose=True,
)



2026-04-22 21:30:36,901 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-22 21:30:36,902 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-22 21:30:36,918 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-22 21:30:36,919 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-22 21:30:36,936 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-22 21:30:36,937 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-22 21:30:36,939 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-22 21:30:36,940 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-22 21:30:37,005 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-22 21:30:37,005 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-22 21:30:37,022 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-22 21:30:37,022 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)



TRANSFORMER | T2 SEQ2ONE | WINDOW_SIZE=L30
targets       = ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
class_weight  = balanced
d_model       = 32
nhead         = 4
batch_size    = 512
optimizer     = adamw
grad_clip     = 1.0
thr_long      = 0.4
thr_short     = 0.4

[BUILD] L30 | n_targets=3


2026-04-22 21:30:37,039 | INFO | Loaded: windows_t2_p40_h60_test.npz
2026-04-22 21:30:37,040 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-22 21:30:37,042 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-22 21:30:37,043 | INFO | Bundle cargado | target=t2_p40_h60 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-22 21:30:37,109 | INFO | Loaded: windows_t2_p50_h30_train.npz
2026-04-22 21:30:37,109 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-22 21:30:37,127 | INFO | Loaded: windows_t2_p50_h30_valid.npz
2026-04-22 21:30:37,127 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-22 21:30:37,144 | INFO | Loaded: windows_t2_p50_h30_test.npz
2026-04-22 21:30:37,144 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-22 21:30:37,147 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-22 21:30:37,147 | INFO | Bundle cargado | target=t2_p50_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | tes


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

[EVAL] VALID | model=transformer_balanced
-> Transformer | L30 | target=t2_p40_h30
-> Transformer | L30 | target=t2_p40_h60
-> Transformer | L30 | target=t2_p50_h30

[DONE] L30
    target  horizon  balanced_accuracy  f1_macro
t2_p40_h30       30           0.403897  0.387970
t2_p40_h60       60           0.410546  0.407382
t2_p50_h30       30           0.407609  0.406756


## **10.5. Métricas**

In [41]:
df_metrics_transformer = results_transformer["metrics"]
df_probabilities_transformer = results_transformer["probabilities"]

In [42]:
print('df_metrics_transformer')
df_metrics_transformer

df_metrics_transformer


,model,split,window_size,target,horizon,n_samples,accuracy,balanced_accuracy,f1_macro,f1_weighted,...,precision_macro_naive,precision_weighted_naive,recall_macro_naive,recall_weighted_naive,balanced_accuracy_gain_vs_naive,f1_macro_gain_vs_naive,f1_weighted_gain_vs_naive,class_weight_mode,optimizer_name,grad_clip_norm
0,transformer_balanced,valid,30,t2_p40_h30,30,6882,0.388986,0.403897,0.387970,0.382312,...,0.131212,0.154949,0.333333,0.393636,0.070563,0.199668,0.159945,balanced,adamw,1.0
1,transformer_balanced,valid,30,t2_p40_h60,60,6882,0.415141,0.410546,0.407382,0.410749,...,0.121137,0.132068,0.333333,0.363412,0.077213,0.229684,0.217017,balanced,adamw,1.0
2,transformer_balanced,valid,30,t2_p50_h30,30,6882,0.412961,0.407609,0.406756,0.410850,...,0.119830,0.129232,0.333333,0.359489,0.074275,0.230470,0.220731,balanced,adamw,1.0


In [43]:
print('df_probabilities_transformer')
df_probabilities_transformer

df_probabilities_transformer


,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,model,target,horizon,class_weight_mode,optimizer_name,grad_clip_norm,threshold_long,threshold_short
0,0.199863,0.548604,0.251533,0,0.548604,1,False,0,False,transformer_balanced,t2_p40_h30,30,balanced,adamw,1.0,0.4,0.4
1,0.225562,0.514413,0.260025,0,0.514413,1,False,0,False,transformer_balanced,t2_p40_h30,30,balanced,adamw,1.0,0.4,0.4
2,0.238733,0.490431,0.270836,0,0.490431,1,False,0,False,transformer_balanced,t2_p40_h30,30,balanced,adamw,1.0,0.4,0.4
3,0.241686,0.489264,0.269051,0,0.489264,1,False,0,False,transformer_balanced,t2_p40_h30,30,balanced,adamw,1.0,0.4,0.4
4,0.228045,0.505648,0.266307,0,0.505648,1,False,0,False,transformer_balanced,t2_p40_h30,30,balanced,adamw,1.0,0.4,0.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20641,0.379985,0.257343,0.362672,-1,0.379985,0,False,0,False,transformer_balanced,t2_p50_h30,30,balanced,adamw,1.0,0.4,0.4
20642,0.373183,0.261272,0.365545,-1,0.373183,0,False,0,False,transformer_balanced,t2_p50_h30,30,balanced,adamw,1.0,0.4,0.4
20643,0.366941,0.257105,0.375954,1,0.375954,0,False,0,False,transformer_balanced,t2_p50_h30,30,balanced,adamw,1.0,0.4,0.4
20644,0.345125,0.271680,0.383195,1,0.383195,0,False,0,False,transformer_balanced,t2_p50_h30,30,balanced,adamw,1.0,0.4,0.4


### Guardado de métricas

In [44]:
save_classification_metrics(
    df_metrics_transformer,
    model_name="transformer",
    split="valid",
)

save_classification_probabilities(
    df_probabilities_transformer,
    model_name="transformer",
    split="valid",
)

2026-04-22 21:31:02,597 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_metrics_transformer_valid.parquet
2026-04-22 21:31:02,635 | INFO | Probabilidades guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_probabilities/classification_probabilities_transformer_valid.parquet


PosixPath('/content/drive/MyDrive/neural_profit/metrics/classification_probabilities/classification_probabilities_transformer_valid.parquet')

## **10.6. Análisis rápido**

In [45]:
def analyze_model_results(
    df_metrics: pd.DataFrame,
    df_probabilities: pd.DataFrame,
) -> None:
    """
    Análisis rápido de resultados de un modelo de clasificación T2.

    Imprime:
    - ranking de métricas
    - resumen operativo
    - precisión en trades
    - conclusión automática
    """

    # =============================
    # 1) Ranking clasificación
    # =============================
    print("\n" + "="*80)
    print("RANKING CLASIFICACIÓN")
    print("="*80)

    cols_cls = [
        "target",
        "horizon",
        "accuracy",
        "balanced_accuracy",
        "balanced_accuracy_gain_vs_naive",
        "f1_macro",
        "f1_weighted",
    ]

    print(
        df_metrics[cols_cls]
        .sort_values("balanced_accuracy", ascending=False)
        .to_string(index=False)
    )

    # =============================
    # 2) Resumen operativo
    # =============================
    print("\n" + "="*80)
    print("RESUMEN OPERATIVO POR TARGET")
    print("="*80)

    summary_probs = (
        df_probabilities
        .groupby(["target", "horizon"], as_index=False)
        .agg(
            n_obs=("target", "size"),
            trade_rate=("trade", "mean"),
            confidence_mean=("confidence", "mean"),
            confidence_median=("confidence", "median"),
            pct_pred_short=("pred_label", lambda s: (s == -1).mean()),
            pct_pred_flat=("pred_label", lambda s: (s == 0).mean()),
            pct_pred_long=("pred_label", lambda s: (s == 1).mean()),
            pct_signal_short=("signal_raw", lambda s: (s == -1).mean()),
            pct_signal_flat=("signal_raw", lambda s: (s == 0).mean()),
            pct_signal_long=("signal_raw", lambda s: (s == 1).mean()),
        )
    )

    print(summary_probs.to_string(index=False))

    # =============================
    # 3) Precisión en trades
    # =============================
    print("\n" + "="*80)
    print("PRECISIÓN SOLO EN TRADES")
    print("="*80)

    trade_only = df_probabilities[df_probabilities["trade"] == True]

    if len(trade_only) == 0:
        print("No hubo trades con los thresholds actuales.")
    else:
        trade_summary = (
            trade_only
            .groupby(["target", "horizon"], as_index=False)
            .agg(
                n_trades=("trade", "size"),
                precision_trades=("is_correct", "mean"),
                confidence_mean_trade=("confidence", "mean"),
                pct_trade_short=("signal_raw", lambda s: (s == -1).mean()),
                pct_trade_long=("signal_raw", lambda s: (s == 1).mean()),
            )
        )
        print(trade_summary.to_string(index=False))

    # =============================
    # 4) Conclusión automática
    # =============================
    print("\n" + "="*80)
    print("CONCLUSIÓN RÁPIDA")
    print("="*80)

    best_target = (
        df_metrics.sort_values("balanced_accuracy", ascending=False)
        .iloc[0]["target"]
    )

    print(f"Mejor target por balanced_accuracy: {best_target}")

    avg_trade_rate = df_probabilities["trade"].mean()
    print(f"Trade rate global: {avg_trade_rate:.4f}")

    if avg_trade_rate < 0.02:
        print("Diagnóstico: el modelo está siendo muy conservador.")
    elif avg_trade_rate < 0.10:
        print("Diagnóstico: el modelo opera poco; revisar thresholds.")
    else:
        print("Diagnóstico: el modelo genera una cantidad razonable de señales.")

In [46]:
analyze_model_results(
    df_metrics_transformer,
    df_probabilities_transformer
)


RANKING CLASIFICACIÓN
    target  horizon  accuracy  balanced_accuracy  balanced_accuracy_gain_vs_naive  f1_macro  f1_weighted
t2_p40_h60       60  0.415141           0.410546                         0.077213  0.407382     0.410749
t2_p50_h30       30  0.412961           0.407609                         0.074275  0.406756     0.410850
t2_p40_h30       30  0.388986           0.403897                         0.070563  0.387970     0.382312

RESUMEN OPERATIVO POR TARGET
    target  horizon  n_obs  trade_rate  confidence_mean  confidence_median  pct_pred_short  pct_pred_flat  pct_pred_long  pct_signal_short  pct_signal_flat  pct_signal_long
t2_p40_h30       30   6882    0.354693         0.432962           0.419485        0.356728       0.396687       0.246585          0.173641         0.645307         0.181052
t2_p40_h60       60   6882    0.317204         0.430557           0.415653        0.315606       0.424005       0.260389          0.164778         0.682796         0.152427
t2_p50_h

## **10.7. Barrido de thresholds**

In [47]:
# =========================================
# Barrido de thresholds para XGBoost
# =========================================

import numpy as np
import pandas as pd


def sweep_thresholds(
    df_probabilities: pd.DataFrame,
    *,
    thresholds: list[float] | None = None,
) -> pd.DataFrame:
    """
    Evalúa múltiples thresholds sobre un DataFrame de probabilidades
    ya generado por un modelo.

    Requiere columnas:
    - target
    - proba_-1
    - proba_1
    - y_true

    Retorna
    -------
    pd.DataFrame con:
    - target
    - threshold
    - trade_rate
    - n_trades
    - precision_trades
    """

    if thresholds is None:
        thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

    results = []

    for thr in thresholds:
        df = df_probabilities.copy()

        # -----------------------------
        # Regla de decisión con threshold
        # -----------------------------
        df["trade_thr"] = (
            (df["proba_1"] >= thr) | (df["proba_-1"] >= thr)
        )

        df["signal_thr"] = 0
        df.loc[df["proba_1"] >= thr, "signal_thr"] = 1
        df.loc[df["proba_-1"] >= thr, "signal_thr"] = -1

        df["correct_thr"] = df["signal_thr"] == df["y_true"]

        # -----------------------------
        # Resumen por target
        # -----------------------------
        summary = (
            df.groupby("target", as_index=False)
            .agg(
                n_obs=("target", "size"),
                n_trades=("trade_thr", "sum"),
                trade_rate=("trade_thr", "mean"),
            )
        )

        # precisión solo en trades
        precision_rows = []
        for target in summary["target"]:
            sub = df[(df["target"] == target) & (df["trade_thr"] == True)]

            if len(sub) > 0:
                precision_trades = sub["correct_thr"].mean()
            else:
                precision_trades = np.nan

            precision_rows.append(precision_trades)

        summary["precision_trades"] = precision_rows
        summary["threshold"] = thr

        results.append(summary)

    df_thresholds = (
        pd.concat(results, ignore_index=True)
        .sort_values(["target", "threshold"])
        .reset_index(drop=True)
    )

    return df_thresholds

In [48]:
df_thresholds_transformer = sweep_thresholds(df_probabilities_transformer)

print(df_thresholds_transformer.to_string(index=False))

    target  n_obs  n_trades  trade_rate  precision_trades  threshold
t2_p40_h30   6882      5492    0.798024          0.375091       0.30
t2_p40_h30   6882      4212    0.612031          0.395537       0.35
t2_p40_h30   6882      2441    0.354693          0.397788       0.40
t2_p40_h30   6882       834    0.121186          0.396882       0.45
t2_p40_h30   6882       253    0.036763          0.367589       0.50
t2_p40_h30   6882        70    0.010171          0.385714       0.55
t2_p40_h60   6882      5542    0.805289          0.348611       0.30
t2_p40_h60   6882      4109    0.597065          0.403018       0.35
t2_p40_h60   6882      2183    0.317204          0.381127       0.40
t2_p40_h60   6882       685    0.099535          0.310949       0.45
t2_p40_h60   6882       170    0.024702          0.205882       0.50
t2_p40_h60   6882        52    0.007556          0.134615       0.55
t2_p50_h30   6882      5502    0.799477          0.342421       0.30
t2_p50_h30   6882      4340    0.6

### **Relación entre threshold, frecuencia de operación y precisión**

**1. Mejor desempeño en clasificación**
   El modelo GRU presenta el mejor desempeño global en términos de `balanced_accuracy` respecto a los modelos previamente evaluados (Logistic Regression, Random Forest y XGBoost).

* El mejor resultado se obtiene en `t2_p40_h60`, con un valor cercano a 0.417, el más alto observado hasta el momento.
* La mejora es consistente en todos los targets evaluados.
* El `gain` respecto al baseline naive es también superior (~0.08).

Esto indica que la incorporación de estructura temporal permite capturar patrones que los modelos tabulares no logran modelar.

**2. Trade-off entre cantidad de señales y calidad**
   Comparado con XGBoost, el GRU muestra un comportamiento más conservador:

* GRU: trade_rate ≈ 0.21
* XGBoost: trade_rate ≈ 0.27

Esto implica que:

* GRU es más selectivo en la generación de señales
* XGBoost es más agresivo

Ambos enfoques son complementarios y pueden ser útiles en un esquema de diversificación de modelos.

**3. Calidad de las señales (trades)**
   Evaluando la precisión únicamente sobre los trades (threshold = 0.40):

* t2_p40_h30 → precision ≈ 0.455
* t2_p50_h30 → precision ≈ 0.432

Comparación con otros modelos:

* Logistic Regression ≈ 0.42–0.45
* Random Forest ≈ 0.40–0.41
* XGBoost ≈ 0.40–0.43
* GRU ≈ 0.43–0.46

El GRU se ubica entre los mejores, con la ventaja adicional de generar menos señales.

**4. Cambio en el target óptimo**
   Se observa un cambio relevante en el target con mejor desempeño:

* Logistic y XGBoost → mejor en `t2_p50_h30`
* GRU → mejor en `t2_p40_h60`

Esto sugiere que el GRU captura mejor dinámicas asociadas a horizontes más largos, lo cual es consistente con su capacidad de modelar dependencias temporales.

**5. Comportamiento frente al threshold**
   El barrido de thresholds muestra una relación clara y estable:

* A menor threshold → mayor trade_rate, menor precisión
* A mayor threshold → menor trade_rate, mayor precisión

Ejemplo en `t2_p40_h30`:

* 0.30 → 76% trades | precision ~0.37
* 0.35 → 46% trades | precision ~0.43
* 0.40 → 18% trades | precision ~0.46
* 0.45 → 3.5% trades | precision ~0.53
* 0.55 → 0.6% trades | precision ~0.73

La curva es suave y monotónica, lo que indica un modelo bien calibrado desde el punto de vista operativo.

**6. Distribución de confianza**
   El nivel medio de confianza se mantiene en torno a 0.41–0.43, similar a otros modelos.

Sin embargo:

* GRU genera menos señales para el mismo threshold
* Esto sugiere probabilidades más concentradas y menor sobreconfianza que modelos basados en árboles

**7. Diagnóstico estructural**
   El modelo cumple con las condiciones necesarias para ser considerado operativo:

* Mejora consistente sobre el baseline naive
* Trade-off controlable mediante thresholds
* Comportamiento estable en múltiples targets

Esto lo posiciona como un candidato sólido para etapas posteriores del pipeline.

**Conclusión**

* GRU presenta el mejor desempeño global en esta etapa de comparación, especialmente en `balanced_accuracy` y estabilidad frente a thresholds.
* XGBoost sigue siendo competitivo, con mayor cantidad de señales y menor complejidad operativa.
* Logistic Regression cumple adecuadamente como baseline, pero queda por debajo en performance.

En esta etapa, GRU y XGBoost emergen como los modelos más relevantes para la comparación final.
